In [ ]:
# Practical 4: autoencoder anomaly detection on ECG (label 1 = normal, 0 = anomaly)
import numpy as np, pandas as pd, tensorflow as tf
from keras import Sequential, layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

data = pd.read_csv('http://storage.googleapis.com/download.tensorflow.org/data/ecg.csv', header=None)
X, y = data.drop(columns=140).values, data[140].astype(int).values
xtr, xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
sc = MinMaxScaler().fit(xtr[ytr == 1])                    # learn "normal" only
xtr, xte = sc.transform(xtr[ytr == 1]), sc.transform(xte)  # train on normal beats only

In [ ]:
# Encoder (140 -> 8 latent) and decoder (8 -> 140)
dense = lambda units: [l for u in units for l in (layers.Dense(u, activation='relu'), layers.Dropout(0.1))]
enc = Sequential([layers.Input((140,)), *dense([64, 32, 16]), layers.Dense(8, activation='relu')])
dec = Sequential([layers.Input((8,)), *dense([16, 32, 64]), layers.Dense(140, activation='sigmoid')])
ae = Sequential([enc, dec])
ae.compile(optimizer='adam', loss='msle', metrics=['mse'])
hist = ae.fit(xtr, xtr, epochs=20, batch_size=512, validation_data=(xte, xte))
pd.DataFrame(hist.history)[['loss', 'val_loss']].plot(xlabel='Epoch', ylabel='MSLE loss');

In [ ]:
# Reconstruction error = anomaly score; threshold = mean + 1 std of error on normal training data
err = lambda a: tf.keras.losses.msle(a, ae.predict(a, verbose=0)).numpy()
e = err(xtr); thr = e.mean() + e.std()
print("Threshold:", thr)
print("Accuracy :", accuracy_score(yte, (err(xte) <= thr).astype(int)))   # low error -> normal (1)